In [1]:
import sys
sys.path.insert(0, '/home/psa14/NEST-rPPG/NEST-rPPG')

In [2]:
import sys
print(sys.executable)

/hpc/group/dunnlab/psa14/conda_envs/rppg-gpu/bin/python


In [3]:
import torch
from model import BaseNet, arc_net

print('PyTorch version:', torch.__version__)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

/hpc/group/dunnlab/psa14/conda_envs/rppg-gpu/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 1.13.1+cu116
Device: cpu


In [4]:
# ST map shape: (batch, RGB channels, skin rows, time frames)
# 9 rows matches STMap.shape = (9, N, 3) from the project
dummy = torch.randn(2, 3, 9, 128).to(device)

base = BaseNet().to(device)
arc  = arc_net().to(device)

base.eval()
arc.eval()

print('BaseNet parameters: {:,}'.format(sum(p.numel() for p in base.parameters())))
print('arc_net parameters: {:,}'.format(sum(p.numel() for p in arc.parameters())))

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /hpc/home/psa14/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:01<00:00, 35.0MB/s]


BaseNet parameters: 13,976,880
arc_net parameters: 15,422,576


In [5]:
with torch.no_grad():
    s1, h1, a1 = base(dummy)
    s2, h2, a2 = arc(dummy)

print('         {:>20}  {:>10}  {:>12}'.format('Sig', 'HR', 'av'))
print('BaseNet  {:>20}  {:>10}  {:>12}'.format(str(tuple(s1.shape)), str(tuple(h1.shape)), str(tuple(a1.shape))))
print('arc_net  {:>20}  {:>10}  {:>12}'.format(str(tuple(s2.shape)), str(tuple(h2.shape)), str(tuple(a2.shape))))

                          Sig          HR            av
BaseNet           (2, 1, 128)      (2, 1)      (2, 960)
arc_net           (2, 1, 128)      (2, 1)      (2, 960)


In [6]:
assert s1.shape == s2.shape, f'Sig mismatch: {s1.shape} vs {s2.shape}'
assert h1.shape == h2.shape, f'HR mismatch:  {h1.shape} vs {h2.shape}'
assert a1.shape == a2.shape, f'av mismatch:  {a1.shape} vs {a2.shape}'

print('✓ All shapes match — arc_net is a valid drop-in replacement for BaseNet')

✓ All shapes match — arc_net is a valid drop-in replacement for BaseNet
